In [14]:
#All Required Libraries
import numpy as np
import random
import sys
import torch
import torch.nn as nn
from IPython.display import HTML, display

print("Imports successful!")

Imports successful!


In [15]:
# Cell 2: Connect 4 Engine State
ROWS = 6
COLS = 7

class Connect4Engine:
    def __init__(self):
        self.board = np.zeros((ROWS, COLS), dtype=int)

    def reset(self):
        self.board = np.zeros((ROWS, COLS), dtype=int)
        return self.board

    def is_valid_col(self, col):
        return self.board[0][col] == 0

    def get_open_row(self, col):
        for r in range(ROWS - 1, -1, -1):
            if self.board[r][col] == 0:
                return r
        return -1

engine = Connect4Engine()
print("Cell 2 Loaded: Engine Ready!")

Cell 2 Loaded: Engine Ready!


In [16]:
# Cell 3: Neural Network Architecture
class Connect4ValNet(nn.Module):
    def __init__(self):
        super(Connect4ValNet, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(ROWS * COLS, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Tanh()
        )

    def forward(self, x):
        return self.fc(x)

model = Connect4ValNet()
model.eval()
print("Cell 3 Loaded: Neural Network Model Initialized!")

Cell 3 Loaded: Neural Network Model Initialized!


In [17]:
# Cell 4: Standalone Self-Rendering HTML/CSS/JS Interface
html_ui = """
<div id="c4-container" style="background-color: #1a1a1a; color: #ffffff; padding: 20px; border-radius: 12px; width: 420px; margin: 0 auto; text-align: center; font-family: sans-serif; box-sizing: border-box;">
    
    <h3 style="margin: 0 0 10px 0; color: #ffffff;">Connect 4: Human vs AI</h3>
    
    <!-- Controls: Difficulty Selector -->
    <div style="margin-bottom: 12px;">
        <label style="font-size: 14px; font-weight: bold; color: #ccc;">Difficulty: </label>
        <select id="diff-select" onchange="setDifficulty(this.value)" style="background: #333; color: #fff; border: 1px solid #555; padding: 4px 8px; border-radius: 4px; font-weight: bold;">
            <option value="easy">Easy 🟢</option>
            <option value="medium">Medium 🟡</option>
            <option value="expert" selected>Expert 🔴</option>
        </select>
    </div>

    <!-- Score Tally Board -->
    <div style="display: flex; justify-content: space-around; background-color: #2a2a2a; padding: 8px; border-radius: 8px; margin-bottom: 12px; border: 1px solid #444;">
        <div>
            <div style="color: #ff4d4d; font-size: 12px; font-weight: bold;">HUMAN</div>
            <div id="tally-human" style="color: #ff4d4d; font-size: 18px; font-weight: bold;">-</div>
        </div>
        <div>
            <div style="color: #ffcc00; font-size: 12px; font-weight: bold;">AI</div>
            <div id="tally-ai" style="color: #ffcc00; font-size: 18px; font-weight: bold;">-</div>
        </div>
        <div>
            <div style="color: #aaaaaa; font-size: 12px; font-weight: bold;">DRAWS</div>
            <div id="tally-draws" style="color: #aaaaaa; font-size: 18px; font-weight: bold;">-</div>
        </div>
    </div>

    <div id="status-line" style="font-size: 15px; font-weight: bold; color: #00e676; height: 22px; margin-bottom: 10px;">Your turn! Click any column.</div>

    <!-- Board Area with Explicit CSS Pixel Grid -->
    <div style="position: relative; width: 378px; height: 324px; margin: 0 auto; background-color: #0d47a1; padding: 8px; border-radius: 10px; box-sizing: content-box;">
        <div id="c4-grid" style="display: flex; flex-direction: column; width: 378px; height: 324px;"></div>
        <svg id="win-svg" style="position: absolute; top: 0; left: 0; width: 100%; height: 100%; pointer-events: none; z-index: 99;"></svg>
    </div>

    <div style="margin-top: 15px;">
        <button onclick="resetBoard()" style="background-color: #00e676; color: #000; border: none; padding: 8px 18px; font-weight: bold; border-radius: 5px; cursor: pointer;">New Game</button>
    </div>
</div>

<script>
(function() {
    const R = 6;
    const C = 7;
    let b = Array(R).fill().map(() => Array(C).fill(0));
    let isOver = false;
    let isThinking = false;
    let difficulty = "expert";
    let scores = { h: 0, a: 0, d: 0 };

    function getTally(n) {
        if (n === 0) return "-";
        let f = Math.floor(n / 5);
        let rem = n % 5;
        let str = "";
        for (let i = 0; i < f; i++) str += "卌 ";
        for (let i = 0; i < rem; i++) str += "|";
        return str;
    }

    function updateTallies() {
        document.getElementById('tally-human').innerText = getTally(scores.h);
        document.getElementById('tally-ai').innerText = getTally(scores.a);
        document.getElementById('tally-draws').innerText = getTally(scores.d);
    }

    window.setDifficulty = function(val) {
        difficulty = val;
    };

    window.buildGrid = function() {
        const grid = document.getElementById('c4-grid');
        grid.innerHTML = '';
        for (let r = 0; r < R; r++) {
            let row = document.createElement('div');
            row.style.display = 'flex';
            row.style.width = '378px';
            row.style.height = '54px';
            for (let c = 0; c < C; c++) {
                let cell = document.createElement('div');
                cell.style.width = '44px';
                cell.style.height = '44px';
                cell.style.backgroundColor = '#111111';
                cell.style.borderRadius = '50%';
                cell.style.margin = '5px';
                cell.style.cursor = 'pointer';
                cell.style.boxSizing = 'border-box';
                cell.id = `cell-${r}-${c}`;
                cell.onclick = function() { window.userClick(c); };
                row.appendChild(cell);
            }
            grid.appendChild(row);
        }
        updateTallies();
    };

    function renderBoard() {
        for (let r = 0; r < R; r++) {
            for (let c = 0; c < C; c++) {
                let el = document.getElementById(`cell-${r}-${c}`);
                if (b[r][c] === 1) el.style.backgroundColor = '#ff4d4d';
                else if (b[r][c] === -1) el.style.backgroundColor = '#ffcc00';
                else el.style.backgroundColor = '#111111';
            }
        }
    }

    function getOpenRow(col) {
        for (let r = R - 1; r >= 0; r--) {
            if (b[r][col] === 0) return r;
        }
        return -1;
    }

    window.userClick = function(col) {
        if (isOver || isThinking) return;
        let r = getOpenRow(col);
        if (r === -1) return;

        isThinking = true;
        b[r][col] = 1;
        renderBoard();

        let win = checkWin(1);
        if (win) {
            drawWinStrike(win);
            document.getElementById('status-line').innerText = "🎉 You Win!";
            document.getElementById('status-line').style.color = "#00e676";
            scores.h++;
            updateTallies();
            isOver = true;
            isThinking = false;
            return;
        }

        if (isFull()) {
            document.getElementById('status-line').innerText = "It's a Draw!";
            scores.d++;
            updateTallies();
            isOver = true;
            isThinking = false;
            return;
        }

        document.getElementById('status-line').innerText = "AI thinking...";
        document.getElementById('status-line').style.color = "#ffcc00";
        setTimeout(aiMove, 300);
    };

    function aiMove() {
        if (isOver) return;
        let col = chooseCol();
        let r = getOpenRow(col);
        b[r][col] = -1;
        renderBoard();

        let win = checkWin(-1);
        if (win) {
            drawWinStrike(win);
            document.getElementById('status-line').innerText = "🤖 AI Wins!";
            document.getElementById('status-line').style.color = "#ff4d4d";
            scores.a++;
            updateTallies();
            isOver = true;
        } else if (isFull()) {
            document.getElementById('status-line').innerText = "It's a Draw!";
            scores.d++;
            updateTallies();
            isOver = true;
        } else {
            document.getElementById('status-text') ? null : null;
            document.getElementById('status-line').innerText = "Your turn! Click any column.";
            document.getElementById('status-line').style.color = "#00e676";
        }
        isThinking = false;
    }

    function chooseCol() {
        let valid = [];
        for (let c = 0; c < C; c++) if (b[0][c] === 0) valid.push(c);

        if (difficulty === "easy") {
            // Random moves with 20% block chance
            for (let c of valid) {
                let r = getOpenRow(c);
                b[r][c] = 1;
                let hWin = checkWin(1);
                b[r][c] = 0;
                if (hWin && Math.random() < 0.2) return c;
            }
            return valid[Math.floor(Math.random() * valid.length)];
        }

        if (difficulty === "medium") {
            // 1. Take win
            for (let c of valid) {
                let r = getOpenRow(c);
                b[r][c] = -1;
                if (checkWin(-1)) { b[r][c] = 0; return c; }
                b[r][c] = 0;
            }
            // 2. Block win
            for (let c of valid) {
                let r = getOpenRow(c);
                b[r][c] = 1;
                if (checkWin(1)) { b[r][c] = 0; return c; }
                b[r][c] = 0;
            }
            return valid[Math.floor(Math.random() * valid.length)];
        }

        // Expert AI
        for (let c of valid) {
            let r = getOpenRow(c);
            b[r][c] = -1;
            if (checkWin(-1)) { b[r][c] = 0; return c; }
            b[r][c] = 0;
        }
        for (let c of valid) {
            let r = getOpenRow(c);
            b[r][c] = 1;
            if (checkWin(1)) { b[r][c] = 0; return c; }
            b[r][c] = 0;
        }
        let pref = [3, 2, 4, 1, 5, 0, 6];
        for (let c of pref) if (valid.includes(c)) return c;
        return valid[0];
    }

    function isFull() {
        return b[0].every(x => x !== 0);
    }

    function checkWin(p) {
        for (let r=0; r<R; r++)
            for (let c=0; c<C-3; c++)
                if (b[r][c]==p && b[r][c+1]==p && b[r][c+2]==p && b[r][c+3]==p) return [[r,c],[r,c+3]];
        for (let r=0; r<R-3; r++)
            for (let c=0; c<C; c++)
                if (b[r][c]==p && b[r+1][c]==p && b[r+2][c]==p && b[r+3][c]==p) return [[r,c],[r+3,c]];
        for (let r=0; r<R-3; r++)
            for (let c=0; c<C-3; c++)
                if (b[r][c]==p && b[r+1][c+1]==p && b[r+2][c+2]==p && b[r+3][c+3]==p) return [[r,c],[r+3,c+3]];
        for (let r=3; r<R; r++)
            for (let c=0; c<C-3; c++)
                if (b[r][c]==p && b[r-1][c+1]==p && b[r-2][c+2]==p && b[r-3][c+3]==p) return [[r,c],[r-3,c+3]];
        return null;
    }

    function drawWinStrike(coords) {
        let x1 = coords[0][1] * 54 + 27 + 8;
        let y1 = coords[0][0] * 54 + 27 + 8;
        let x2 = coords[1][1] * 54 + 27 + 8;
        let y2 = coords[1][0] * 54 + 27 + 8;
        let svg = document.getElementById('win-svg');
        svg.innerHTML = `<line x1="${x1}" y1="${y1}" x2="${x2}" y2="${y2}" stroke="#00e676" stroke-width="8" stroke-linecap="round" />`;
    }

    window.resetBoard = function() {
        b = Array(R).fill().map(() => Array(C).fill(0));
        isOver = false;
        isThinking = false;
        document.getElementById('win-svg').innerHTML = '';
        document.getElementById('status-line').innerText = "Your turn! Click any column.";
        document.getElementById('status-line').style.color = "#00e676";
        renderBoard();
    };

    setTimeout(window.buildGrid, 100);
})();
</script>
"""

display(HTML(html_ui))